In [4]:
import pyodbc
import pandas as pd

In [7]:
SERVER = 'BPD05REDAWS.redfactor.com.br'
DATABASE = 'HOMOLOGACAO_PD'
UID = 'sys_r_bi'
PWD = 'sysrbi123'


In [8]:
conn = pyodbc.connect( 
        "Driver={ODBC Driver 17 for SQL Server};"
        f"Server={SERVER};"
        f"UID={UID};"
        f"PWD={PWD};"
        "encoding=UTF-8;"
        f"Database={DATABASE};"
    )

Error: ('01000', "[01000] [unixODBC][Driver Manager]Can't open lib 'ODBC Driver 17 for SQL Server' : file not found (0) (SQLDriverConnect)")

In [ ]:
import pyodbc
import pandas as pd

# Conexão com o banco de ORIGEM


# Conexão com o banco de DESTINO
conn_destino = pyodbc.connect(
    'Driver={ODBC Driver 17 for SQL Server};'
    'Server=BPD04REDAWS.redfactor.com.br;'
    'Database=PRODUTIVO_PD;'
    'UID=sys_r_bi;'
    'PWD=sysrbi123;'
    'encoding=UTF-8;'
)
cursor_destino = conn_destino.cursor()

# Nome das tabelas
tabela_origem = 'PAINEL_CONFIRMACAO_PRODUTIVIDADE_CONFIRMACAO_PD'
tabela_destino = 'PAINEL_CONFIRMACAO_PRODUTIVIDADE_CONFIRMACAO_PD_BKP'

# Tamanho dos blocos de leitura
tamanho_chunk = 1000
offset = 0
total_inseridos = 0

while True:
    print(f"🔄 Processando linhas {offset} a {offset + tamanho_chunk - 1}")

    query = f"""
        SELECT *
        FROM {tabela_origem}
        ORDER BY ID -- ajuste para uma coluna que pode ordenar (ex: ID ou data)
        OFFSET {offset} ROWS
        FETCH NEXT {tamanho_chunk} ROWS ONLY
    """

    df_chunk = pd.read_sql(query, conn_origem)

    if df_chunk.empty:
        print("✅ Fim dos dados.")
        break

    # Inserir dados no banco de destino
    colunas = ', '.join(df_chunk.columns)
    placeholders = ', '.join(['?'] * len(df_chunk.columns))

    for _, row in df_chunk.iterrows():
        valores = tuple(row)
        sql_insert = f"INSERT INTO {tabela_destino} ({colunas}) VALUES ({placeholders})"
        cursor_destino.execute(sql_insert, valores)

    conn_destino.commit()
    total_inseridos += len(df_chunk)
    print(f"✅ {len(df_chunk)} registros inseridos. Total: {total_inseridos}")

    offset += tamanho_chunk

# Fechando conexões
cursor_destino.close()
conn_origem.close()
conn_destino.close()
print("🏁 Processo finalizado.")


Error: ('01000', "[01000] [unixODBC][Driver Manager]Can't open lib 'ODBC Driver 17 for SQL Server' : file not found (0) (SQLDriverConnect)")